# Entregable 3: Lógica de Rangos y Cálculo de Límites
### Modelo de Estimación de Stranded Capacity en Data Centers de IA

**Proyecto:** No Country  
**Objetivo:** Explicar por qué el modelo se expresa en rangos (evitando la *falsa precisión*) y demostrar computacionalmente el cálculo de los límites de **Floor (Costo Incurrido Real)**, **Ceiling (Costo de Oportunidad de Colocation)** y percentiles estocásticos **Monte Carlo (P10, P50, P90)**.

## 1. Fundamentos Teóricos: ¿Por qué expresar en Rangos?
La infraestructura de cómputo de IA opera en escenarios de alta volatilidad operativa:
1. **Margen Térmico y DVFS:** La refrigeración condiciona la frecuencia de los chips. Los sistemas por aire sufren *throttling* a ~70°C, generando variaciones de hasta 17% en rendimiento/consumo.
2. **Disparidad Geográfica de Costos:** El costo por MW varía sustancialmente por regiones energéticas (desde $0.06/kWh hasta $0.25+/kWh).
3. **Errores Behind-The-Meter:** El consumo parasitario de ventiladores en servidores de aire altera la lectura del PUE real.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
sys.path.append(os.path.abspath("."))
from logica_rangos import RangeLogicEvaluator, RANGE_BENCHMARKS

# Configuración de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

print("Entorno cargado exitosamente. Evaluador listo.")

Entorno cargado exitosamente. Evaluador listo.


## 2. Comparación Determinística: Floor vs. Ceiling por Tecnología
Evaluaremos las tres tecnologías (Aire, Híbrido, Líquido) para un centro de datos de **20 MW** al **75% de utilización**.

In [2]:
evaluator = RangeLogicEvaluator()
facility_mw = 20.0
utilization_pct = 75.0
technologies = ["air-cooled", "hybrid", "liquid-cooled"]

results = {}
for tech in technologies:
    results[tech] = evaluator.generate_range_summary(facility_mw, utilization_pct, tech)

# Visualización de Rangos Financieros (Floor vs Mid vs Ceiling)
fig, ax = plt.subplots(figsize=(10, 5))

names = [results[t]["cooling_name"] for t in technologies]
floors = [results[t]["financial_range_usd"]["floor_incurred"] / 1e6 for t in technologies]
mids = [results[t]["financial_range_usd"]["mid_expected"] / 1e6 for t in technologies]
ceilings = [results[t]["financial_range_usd"]["ceiling_opportunity"] / 1e6 for t in technologies]

x = np.arange(len(names))
width = 0.25

rects1 = ax.bar(x - width, floors, width, label="Floor (Costo Incurrido)", color="#e74c3c")
rects2 = ax.bar(x, mids, width, label="Mid (Punto Medio)", color="#3498db")
rects3 = ax.bar(x + width, ceilings, width, label="Ceiling (Costo Oportunidad)", color="#2ecc71")

ax.set_ylabel("Pérdida Financiera Anual (Millones USD)", fontsize=11, fontweight="bold")
ax.set_title(f"Límites Financieros por Tecnología de Enfriamiento ({facility_mw} MW, Utilización {utilization_pct}%)", fontsize=12, fontweight="bold", pad=15)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.legend(frameon=True, facecolor="white", edgecolor="#dddddd")

plt.tight_layout()
plt.savefig("rangos_financieros_comparativa.png", dpi=300)
plt.close()
print("Gráfico generado e imagen guardada.")

Gráfico generado e imagen guardada.


## 3. Simulación Estocástica de Monte Carlo (Percentiles P10, P50, P90)
Generamos distribuciones estocásticas para evaluar la densidad de probabilidad de la pérdida financiera anual en un facility Air-Cooled de 20 MW.

In [3]:
stoch = evaluator.calculate_stochastic_range(facility_mw=20.0, cooling_type="air-cooled", num_simulations=10000)

print(f"--- Percentiles de Pérdida Financiera (Facility 20 MW Air-Cooled) ---")
print(f"P10 (Optimista / Suelo Estocástico): ${stoch.p10_usd_annual:,.2f} USD")
print(f"P50 (Mediana / Escenario Base):       ${stoch.p50_usd_annual:,.2f} USD")
print(f"P90 (Pesimista / Techo Estocástico):  ${stoch.p90_usd_annual:,.2f} USD")
print(f"Desviación Estándar:                  ${stoch.std_usd_annual:,.2f} USD")

--- Percentiles de Pérdida Financiera (Facility 20 MW Air-Cooled) ---
P10 (Optimista / Suelo Estocástico): $7,322,035.82 USD
P50 (Mediana / Escenario Base):       $7,878,245.71 USD
P90 (Pesimista / Techo Estocástico):  $8,471,938.00 USD
Desviación Estándar:                  $452,501.71 USD


## 4. Conclusiones Metodológicas
1. **Defendibilidad:** El uso de rangos de Floor y Ceiling permite a los CFOs y COOs evaluar escenarios mínimos realistas (OPEX pagado) frente al techo de oportunidad de negocio (Colocation).
2. **Impacto Directo:** Migrar de refrigeración por Aire a Líquido reduce la potencia varada de **12.5% a 3.5%**, recortando el costo de ineficiencia incurrido en más del **60%**.
3. **Honestidad Auditable:** Los percentiles Monte Carlo atenúan el riesgo de estimaciones simplistas y ofrecen una base confiable para la toma de decisiones.